In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
os.environ["CUDA_DEVICE_ORDER"]="PCI_BUS_ID"   # see issue #152
os.environ["CUDA_VISIBLE_DEVICES"]="4"

In [ ]:
import torch
import numpy as np

In [ ]:
from aidan_lib.models.sam3_video import SAM3Harness

In [ ]:
from pathlib import Path
import cv2
import imageio
from PIL import Image

In [ ]:
from aidan_lib.definitions import DATA_DIR
test_vid_path = DATA_DIR / "tip_to_tip_short.mp4"
assert test_vid_path.exists(), f"Test video does not exist {test_vid_path.absolute().as_posix()}"
print(test_vid_path)

In [ ]:
from aidan_lib.video_utils.load_batched_frames import load_batched_frames, load_constrained_batched_frames
from aidan_lib.video_utils.scene_split import get_constrained_scenes, get_transnet_model

In [ ]:
transnet = get_transnet_model("cuda")
constrained_scenes = get_constrained_scenes(test_vid_path, transnet, threshold=0.75)

In [ ]:
harness = SAM3Harness(max_num_objects=64)

In [ ]:
batch_frame_loader = load_constrained_batched_frames(test_vid_path, constrained_scenes, batch_size=120, skip_frames=5, convert_pil=True, overlap=1)

In [ ]:
from aidan_lib.models.sam3_video import generate_video_segmentation
from aidan_lib.visualization.segmentations import visualize_segmentations, int_mask_to_binary_masks
import numpy as np

In [ ]:
prompts = ["Person", "Light"]
frame_seg_generator = generate_video_segmentation(
    harness=harness, 
    prompts=prompts, 
    batch_frame_loader=batch_frame_loader
)

In [ ]:
output_path = test_vid_path.parent / f"{test_vid_path.stem}_w_broad_segs.mp4"
print(output_path)

In [ ]:
fps = 15
from tqdm import tqdm

# Use imageio"s writer in a context manager to ensure it closes properly
progress = tqdm()
with imageio.get_writer(output_path, fps=fps, format="mp4", codec="libx264") as writer:
    for frame_info in frame_seg_generator:
        # Unpack the FrameSegmentationInfo (removed obj_id_to_prompt)
        frame_num, frame, frame_segmentations, background_index = frame_info
        
        # Combine all masks and object IDs from each prompt
        all_masks = []
        all_obj_ids = []
        obj_prompts = []
        for prompt, sam_seg in frame_segmentations.items():
            masks, obj_ids = int_mask_to_binary_masks(sam_seg, background_index=background_index)
            all_masks.extend(masks)
            all_obj_ids.extend(obj_ids)
            # Duplicate the prompt string for each object ID detected for this prompt
            obj_prompts.extend([prompt] * len(obj_ids))
            
        # Generate labels dynamically
        labels = [f"{obj_prompt} {obj_id}" for obj_prompt, obj_id in zip(obj_prompts, all_obj_ids)]
        
        # Create the visualization (img is a PIL Image)
        img = visualize_segmentations(frame, all_masks, labels=labels)
        
        # Convert the PIL Image to a NumPy array for imageio
        frame_array = np.expand_dims(np.array(img), axis=0)
        
        # Write the frame to the video file
        writer.append_data(frame_array)
        
        progress.update(1)
        progress.set_description(f"Frame {frame_num}")


print(f"Video saved successfully to {output_path.absolute()}")